# The Transformer Block (GPT-2 Style)

The transformer block is the fundamental repeating unit of GPT-2. Each block takes a sequence of token representations and refines them through two sub-layers:

1. **Multi-head causal self-attention** -- lets each token attend to all preceding tokens.
2. **Position-wise feed-forward network** -- applies a non-linear transformation independently to each position.

GPT-2 uses a **pre-norm** architecture: LayerNorm is applied *before* each sub-layer rather than after. Both sub-layers are wrapped in **residual connections** so the input is added back to the output.

The block structure looks like this:

```
x ──┬── LayerNorm ── MultiHeadAttn ──┐
    └────────────────────────────────+──┬── LayerNorm ── FFN ──┐
                                       └─────────────────────+── output
```

In this notebook we will:
- Build **LayerNorm** from scratch and verify it against PyTorch
- Implement the **GELU** activation and compare it with ReLU
- Understand **residual connections** and their impact on gradient flow
- Assemble a complete **TransformerBlock** (with inline multi-head attention)
- Visualize activations, norms, and distributions throughout the block
- Run **ablation experiments** -- removing residuals, removing LayerNorm, swapping GELU for ReLU

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Tiny dimensions for fast experimentation
EMBED_DIM = 64
N_HEADS = 4
SEQ_LEN = 16
BATCH_SIZE = 2

print(f"embed_dim={EMBED_DIM}, n_heads={N_HEADS}, head_dim={EMBED_DIM // N_HEADS}")
print(f"seq_len={SEQ_LEN}, batch_size={BATCH_SIZE}")

---
## Part 1: LayerNorm from Scratch

Layer Normalization normalizes each sample independently across the feature dimension. For a vector $\mathbf{x} \in \mathbb{R}^d$:

$$\text{LayerNorm}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are the mean and variance of $\mathbf{x}$ across the feature dimension, and $\gamma$, $\beta$ are learnable parameters.

GPT-2 uses **pre-norm**: LayerNorm is applied *before* each sub-layer, which tends to stabilize training compared to post-norm.

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm implemented from scratch."""

    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # Compute mean and variance across the last dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        # Scale and shift with learnable parameters
        return self.gamma * x_norm + self.beta


print("LayerNorm module defined.")

In [ ]:
# Verify our LayerNorm matches PyTorch's nn.LayerNorm
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

our_ln = LayerNorm(EMBED_DIM)
ref_ln = nn.LayerNorm(EMBED_DIM)

# Copy weights so both use the same gamma/beta
ref_ln.weight.data.copy_(our_ln.gamma.data)
ref_ln.bias.data.copy_(our_ln.beta.data)

out_ours = our_ln(x)
out_ref = ref_ln(x)

max_diff = (out_ours - out_ref).abs().max().item()
print(f"Max absolute difference: {max_diff:.2e}")
assert max_diff < 1e-5, "Mismatch!"
print("Our LayerNorm matches nn.LayerNorm.")

# Show input vs output statistics
print("=== Input statistics (per-sample, per-position) ===")
print(f"  Mean of means:  {x.mean(dim=-1).mean().item():.4f}")
print(f"  Mean of stdevs: {x.std(dim=-1).mean().item():.4f}")

print("\n=== Output statistics after LayerNorm ===")
print(f"  Mean of means:  {out_ours.mean(dim=-1).mean().item():.6f}  (should be ~0)")
print(f"  Mean of stdevs: {out_ours.std(dim=-1).mean().item():.4f}  (should be ~1)")

LayerNorm centers each token's feature vector to zero mean and unit variance, then allows the network to learn an optimal scale and shift. This stabilizes activations regardless of what the upstream layers produce.

---
## Part 2: Feed-Forward Network with GELU

The position-wise feed-forward network in GPT-2 consists of two linear layers with a GELU activation in between. The hidden dimension is 4x the embedding dimension:

$$\text{FFN}(\mathbf{x}) = W_2 \, \text{GELU}(W_1 \mathbf{x} + b_1) + b_2$$

GELU (Gaussian Error Linear Unit) is a smooth approximation of ReLU that allows small negative values to pass through:

$$\text{GELU}(x) = x \cdot \frac{1}{2}\left[1 + \tanh\left(\sqrt{\frac{2}{\pi}}\left(x + 0.044715\,x^3\right)\right)\right]$$

In [ ]:
def gelu(x):
    """GELU activation -- tanh approximation used in GPT-2."""
    return x * 0.5 * (1.0 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x ** 3)
    ))


# Quick check: GELU(0) should be 0
print(f"GELU(0) = {gelu(torch.tensor(0.0)).item():.6f}")
print(f"GELU(1) = {gelu(torch.tensor(1.0)).item():.6f}")
print(f"GELU(-1) = {gelu(torch.tensor(-1.0)).item():.6f}  (small but nonzero -- unlike ReLU)")

In [ ]:
# Plot GELU vs ReLU
xs = torch.linspace(-4, 4, 500)
y_gelu = gelu(xs)
y_relu = F.relu(xs)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(xs.numpy(), y_gelu.numpy(), label="GELU", linewidth=2)
ax.plot(xs.numpy(), y_relu.numpy(), label="ReLU", linewidth=2, linestyle="--")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Input")
ax.set_ylabel("Output")
ax.set_title("GELU vs ReLU Activation Functions")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Key differences between GELU and ReLU:**
- GELU is smooth everywhere -- no hard kink at zero, so gradients are smoother.
- GELU allows small negative values to pass through, avoiding the "dying neuron" problem.
- For large positive inputs, GELU behaves almost identically to ReLU.
- The smooth transition near zero provides a probabilistic gating effect.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network with GELU activation."""

    def __init__(self, embed_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 4 * embed_dim)
        self.fc2 = nn.Linear(4 * embed_dim, embed_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = gelu(x)
        x = self.fc2(x)
        return x


ffn = FeedForward(EMBED_DIM)
n_params = sum(p.numel() for p in ffn.parameters())
print(f"FFN parameters: {n_params:,}")
print(f"  fc1: {EMBED_DIM} -> {4 * EMBED_DIM}  ({EMBED_DIM * 4 * EMBED_DIM + 4 * EMBED_DIM:,} params)")
print(f"  fc2: {4 * EMBED_DIM} -> {EMBED_DIM}  ({4 * EMBED_DIM * EMBED_DIM + EMBED_DIM:,} params)")

# Run FFN on our input
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
ffn_out = ffn(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {ffn_out.shape}")
print(f"Input  -- mean: {x.mean().item():.4f}, std: {x.std().item():.4f}")
print(f"Output -- mean: {ffn_out.mean().item():.4f}, std: {ffn_out.std().item():.4f}")

---
## Part 3: Residual Connections

Residual connections (skip connections) are critical for training deep networks. The idea is simple:

$$\text{output} = \mathbf{x} + \text{sublayer}(\mathbf{x})$$

This helps gradient flow because during backpropagation:

$$\frac{\partial \text{output}}{\partial \mathbf{x}} = I + \frac{\partial \text{sublayer}(\mathbf{x})}{\partial \mathbf{x}}$$

The identity term $I$ ensures gradients always have a direct path back, even if the sublayer's gradients are small. Without residual connections, gradients must pass through every transformation, and they tend to vanish in deep networks.

In [ ]:
# Demonstrate residual connections
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)

# Without residual: just pass through the FFN
out_no_res = ffn(x)
loss_no_res = out_no_res.sum()
loss_no_res.backward()
grad_no_res = x.grad.clone()

# With residual: x + FFN(x)
x.grad = None  # Reset gradients
out_with_res = x + ffn(x)
loss_with_res = out_with_res.sum()
loss_with_res.backward()
grad_with_res = x.grad.clone()

print("Gradient statistics:")
print(f"  Without residual -- mean: {grad_no_res.abs().mean().item():.6f}, "
      f"std: {grad_no_res.std().item():.6f}")
print(f"  With residual    -- mean: {grad_with_res.abs().mean().item():.6f}, "
      f"std: {grad_with_res.std().item():.6f}")
print(f"\nGradient magnitude ratio (with/without): "
      f"{grad_with_res.abs().mean().item() / grad_no_res.abs().mean().item():.2f}x")

In [ ]:
# Simulate gradient flow through N stacked layers with and without residuals
torch.manual_seed(42)
n_layers = 12

layers = [FeedForward(EMBED_DIM) for _ in range(n_layers)]

# Without residuals
x_no_res = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
h = x_no_res
for layer in layers:
    h = layer(h)
h.sum().backward()
grad_norm_no_res = x_no_res.grad.norm().item()

# With residuals
x_with_res = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
# Use same initial values
x_with_res.data.copy_(x_no_res.data)
h = x_with_res
for layer in layers:
    h = h + layer(h)
h.sum().backward()
grad_norm_with_res = x_with_res.grad.norm().item()

print(f"After {n_layers} stacked layers:")
print(f"  Gradient norm WITHOUT residuals: {grad_norm_no_res:.6f}")
print(f"  Gradient norm WITH residuals:    {grad_norm_with_res:.6f}")
print(f"  Ratio: {grad_norm_with_res / (grad_norm_no_res + 1e-10):.1f}x")

Residual connections provide a gradient "highway" that prevents vanishing gradients. Even with 12 stacked layers, the gradient norm with residuals remains healthy, while without residuals it can shrink dramatically.

---
## Part 4: Multi-Head Causal Self-Attention (Inline)

Since this notebook is self-contained, we re-implement multi-head attention here with a causal mask. Each head independently computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

where $M$ is a causal mask that sets future positions to $-\infty$.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads

        # Combined projection for Q, K, V
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape

        # Project to Q, K, V
        qkv = self.qkv_proj(x)  # (B, T, 3*C)
        q, k, v = qkv.split(self.embed_dim, dim=-1)  # each (B, T, C)

        # Reshape for multi-head: (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        scale = self.head_dim ** 0.5
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, n_heads, T, T)

        # Causal mask: prevent attending to future positions
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn_scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, n_heads, T, T)

        # Weighted sum of values
        attn_out = torch.matmul(attn_weights, v)  # (B, n_heads, T, head_dim)

        # Concatenate heads and project
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(attn_out)


# Test it
torch.manual_seed(42)
mha = MultiHeadAttention(EMBED_DIM, N_HEADS)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
attn_out = mha(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {attn_out.shape}")
print(f"MHA parameters: {sum(p.numel() for p in mha.parameters()):,}")

---
## Part 5: Assembling the Full Transformer Block

Now we combine all components into the GPT-2 pre-norm transformer block:

```
x -> LayerNorm -> MultiHeadAttn -> + (residual) -> LayerNorm -> FFN -> + (residual) -> output
     ^                             |                ^                   |
     └─────────────────────────────┘                └───────────────────┘
```

In [ ]:
class TransformerBlock(nn.Module):
    """GPT-2 style transformer block with pre-norm."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        # Pre-norm attention with residual
        x = x + self.attn(self.ln1(x))
        # Pre-norm FFN with residual
        x = x + self.ffn(self.ln2(x))
        return x


block = TransformerBlock(EMBED_DIM, N_HEADS)
n_params = sum(p.numel() for p in block.parameters())
print(f"TransformerBlock total parameters: {n_params:,}")
print(f"\nBreakdown:")
for name, module in block.named_children():
    mp = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {mp:,} params")

In [ ]:
# Run the full block
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
out = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"\nInput  stats -- mean: {x.mean().item():.4f}, std: {x.std().item():.4f}, "
      f"norm: {x.norm().item():.2f}")
print(f"Output stats -- mean: {out.mean().item():.4f}, std: {out.std().item():.4f}, "
      f"norm: {out.norm().item():.2f}")

The block preserves the shape of its input exactly. Thanks to residual connections, the output norm stays in a similar range to the input norm -- the block makes a modest refinement rather than a drastic transformation.

---
## Part 6: Visualizations

### 6.1 Activation Distributions Before and After LayerNorm

In [ ]:
# Create input with non-standard distribution (shifted and scaled)
torch.manual_seed(42)
x_raw = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM) * 3.0 + 2.0  # mean~2, std~3
ln = nn.LayerNorm(EMBED_DIM)
x_normed = ln(x_raw)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

# Flatten for histograms
raw_vals = x_raw.detach().numpy().flatten()
normed_vals = x_normed.detach().numpy().flatten()

ax.hist(raw_vals, bins=80, alpha=0.6, label="Before LayerNorm", density=True, color="steelblue")
ax.hist(normed_vals, bins=80, alpha=0.6, label="After LayerNorm", density=True, color="coral")
ax.set_xlabel("Activation Value")
ax.set_ylabel("Density")
ax.set_title("Activation Distributions Before and After LayerNorm")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Before -- mean: {raw_vals.mean():.3f}, std: {raw_vals.std():.3f}")
print(f"After  -- mean: {normed_vals.mean():.4f}, std: {normed_vals.std():.4f}")

LayerNorm centers the distribution around zero and normalizes the spread to approximately unit variance, regardless of the input's original statistics.

### 6.2 GELU vs ReLU -- Derivative Comparison

In [ ]:
# Plot the activations AND their derivatives side by side
xs = torch.linspace(-4, 4, 500, requires_grad=False)

# Compute GELU and its derivative
xs_g = xs.clone().requires_grad_(True)
y_gelu = gelu(xs_g)
gelu_grad = torch.autograd.grad(y_gelu.sum(), xs_g)[0]

# Compute ReLU derivative manually (0 for x<0, 1 for x>0)
relu_grad = (xs > 0).float()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Activations
axes[0].plot(xs.numpy(), gelu(xs).detach().numpy(), label="GELU", linewidth=2)
axes[0].plot(xs.numpy(), F.relu(xs).numpy(), label="ReLU", linewidth=2, linestyle="--")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].axvline(0, color="gray", linewidth=0.5)
axes[0].set_title("Activation Functions")
axes[0].set_xlabel("Input")
axes[0].set_ylabel("Output")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Derivatives
axes[1].plot(xs.numpy(), gelu_grad.detach().numpy(), label="GELU'", linewidth=2)
axes[1].plot(xs.numpy(), relu_grad.numpy(), label="ReLU'", linewidth=2, linestyle="--")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].axvline(0, color="gray", linewidth=0.5)
axes[1].set_title("Derivatives")
axes[1].set_xlabel("Input")
axes[1].set_ylabel("Gradient")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The GELU derivative is smooth and continuous, while ReLU has a hard discontinuity at zero with zero gradient for all negative inputs. GELU's smooth derivative enables more stable optimization.

### 6.3 Residual Stream Norms Through the Block

In [ ]:
# Track the norm of the residual stream at each stage inside the block
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

stages = []
norms = []

# Stage 0: Input
stages.append("Input")
norms.append(x.norm(dim=-1).mean().item())

# Stage 1: After LN1
h_ln1 = block.ln1(x)
stages.append("After LN1")
norms.append(h_ln1.norm(dim=-1).mean().item())

# Stage 2: After Attention (before residual)
h_attn = block.attn(h_ln1)
stages.append("Attn Output")
norms.append(h_attn.norm(dim=-1).mean().item())

# Stage 3: After first residual
h_res1 = x + h_attn
stages.append("+ Residual 1")
norms.append(h_res1.norm(dim=-1).mean().item())

# Stage 4: After LN2
h_ln2 = block.ln2(h_res1)
stages.append("After LN2")
norms.append(h_ln2.norm(dim=-1).mean().item())

# Stage 5: After FFN (before residual)
h_ffn = block.ffn(h_ln2)
stages.append("FFN Output")
norms.append(h_ffn.norm(dim=-1).mean().item())

# Stage 6: After second residual (final output)
h_res2 = h_res1 + h_ffn
stages.append("+ Residual 2\n(Output)")
norms.append(h_res2.norm(dim=-1).mean().item())

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#55A868", "#C44E52", "#8172B2"]
bars = ax.bar(range(len(stages)), norms, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(stages)))
ax.set_xticklabels(stages, fontsize=9)
ax.set_ylabel("Mean L2 Norm (per token)")
ax.set_title("Residual Stream Norm at Each Stage of the Transformer Block")
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar, val in zip(bars, norms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

Notice how LayerNorm normalizes the representation to a consistent norm, and the residual connections ensure the overall norm grows gradually rather than exploding or collapsing.

---
## Part 7: Ablation Experiments

To understand *why* each component matters, let us systematically remove or replace them.

### 7.1 Ablation: Remove Residual Connections

In [ ]:
class TransformerBlockNoResidual(nn.Module):
    """Transformer block WITHOUT residual connections."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        # No residual -- just sequential
        x = self.attn(self.ln1(x))
        x = self.ffn(self.ln2(x))
        return x


print("TransformerBlockNoResidual defined.")

In [ ]:
# Compare gradient norms through N stacked blocks
torch.manual_seed(42)
n_stack = 6

# With residuals
blocks_res = nn.ModuleList([TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(n_stack)])
x_res = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
h = x_res
for b in blocks_res:
    h = b(h)
h.sum().backward()
grad_res = x_res.grad.norm().item()

# Without residuals
blocks_nores = nn.ModuleList([TransformerBlockNoResidual(EMBED_DIM, N_HEADS) for _ in range(n_stack)])
x_nores = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
h = x_nores
for b in blocks_nores:
    h = b(h)
h.sum().backward()
grad_nores = x_nores.grad.norm().item()

print(f"Gradient norm through {n_stack} blocks:")
print(f"  WITH residuals:    {grad_res:.4f}")
print(f"  WITHOUT residuals: {grad_nores:.4f}")
print(f"  Ratio (with/without): {grad_res / (grad_nores + 1e-10):.1f}x")

In [ ]:
# Visualize how gradient norms change with depth
torch.manual_seed(42)
max_depth = 8

grad_norms_res = []
grad_norms_nores = []

for depth in range(1, max_depth + 1):
    # With residuals
    blocks_r = nn.ModuleList([TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(depth)])
    xr = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
    h = xr
    for b in blocks_r:
        h = b(h)
    h.sum().backward()
    grad_norms_res.append(xr.grad.norm().item())

    # Without residuals
    blocks_nr = nn.ModuleList([TransformerBlockNoResidual(EMBED_DIM, N_HEADS) for _ in range(depth)])
    xnr = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
    h = xnr
    for b in blocks_nr:
        h = b(h)
    h.sum().backward()
    grad_norms_nores.append(xnr.grad.norm().item())

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
depths = list(range(1, max_depth + 1))
ax.plot(depths, grad_norms_res, "o-", label="With Residuals", linewidth=2, markersize=6)
ax.plot(depths, grad_norms_nores, "s--", label="Without Residuals", linewidth=2, markersize=6)
ax.set_xlabel("Number of Stacked Blocks")
ax.set_ylabel("Input Gradient Norm")
ax.set_title("Gradient Norm vs Depth: With and Without Residual Connections")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

Without residual connections, the gradient norm tends to decay or fluctuate unpredictably as depth increases. With residuals, the gradient norm remains far more stable, enabling effective training of deep transformers.

### 7.2 Ablation: Remove LayerNorm

In [ ]:
class TransformerBlockNoNorm(nn.Module):
    """Transformer block WITHOUT LayerNorm."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(x)   # No LN before attention
        x = x + self.ffn(x)    # No LN before FFN
        return x


print("TransformerBlockNoNorm defined.")

In [ ]:
# Track activation distribution drift through stacked blocks
torch.manual_seed(42)
n_stack = 6

blocks_norm = nn.ModuleList([TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(n_stack)])
blocks_nonorm = nn.ModuleList([TransformerBlockNoNorm(EMBED_DIM, N_HEADS) for _ in range(n_stack)])

x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

stats_norm = [(x.mean().item(), x.std().item())]
stats_nonorm = [(x.mean().item(), x.std().item())]

h_norm = x.clone()
h_nonorm = x.clone()

for i in range(n_stack):
    h_norm = blocks_norm[i](h_norm)
    h_nonorm = blocks_nonorm[i](h_nonorm)
    stats_norm.append((h_norm.mean().item(), h_norm.std().item()))
    stats_nonorm.append((h_nonorm.mean().item(), h_nonorm.std().item()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
layer_idx = list(range(n_stack + 1))

# Mean
axes[0].plot(layer_idx, [s[0] for s in stats_norm], "o-", label="With LayerNorm", linewidth=2)
axes[0].plot(layer_idx, [s[0] for s in stats_nonorm], "s--", label="Without LayerNorm", linewidth=2)
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Mean Activation")
axes[0].set_title("Activation Mean Drift")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Std
axes[1].plot(layer_idx, [s[1] for s in stats_norm], "o-", label="With LayerNorm", linewidth=2)
axes[1].plot(layer_idx, [s[1] for s in stats_nonorm], "s--", label="Without LayerNorm", linewidth=2)
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Std of Activations")
axes[1].set_title("Activation Std Drift")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Without LayerNorm, activation statistics drift as we go deeper -- the standard deviation can grow or shift, making optimization increasingly difficult. LayerNorm keeps the activations well-behaved at every layer.

### 7.3 Ablation: GELU vs ReLU in the Transformer Block

In [ ]:
class TransformerBlockReLU(nn.Module):
    """Transformer block with ReLU instead of GELU."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


print("TransformerBlockReLU defined.")

In [ ]:
# Compare output distributions: GELU block vs ReLU block
torch.manual_seed(42)
block_gelu = TransformerBlock(EMBED_DIM, N_HEADS)
block_relu = TransformerBlockReLU(EMBED_DIM, N_HEADS)

# Copy weights from GELU block to ReLU block (except the activation)
block_relu.ln1.load_state_dict(block_gelu.ln1.state_dict())
block_relu.attn.load_state_dict(block_gelu.attn.state_dict())
block_relu.ln2.load_state_dict(block_gelu.ln2.state_dict())
# Copy FFN linear layers (indices 0 and 2 in the Sequential)
block_relu.ffn[0].load_state_dict(block_gelu.ffn[0].state_dict())
block_relu.ffn[2].load_state_dict(block_gelu.ffn[2].state_dict())

x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
out_gelu = block_gelu(x).detach().numpy().flatten()
out_relu = block_relu(x).detach().numpy().flatten()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(out_gelu, bins=80, alpha=0.6, label="GELU Block", density=True, color="steelblue")
ax.hist(out_relu, bins=80, alpha=0.6, label="ReLU Block", density=True, color="coral")
ax.set_xlabel("Output Value")
ax.set_ylabel("Density")
ax.set_title("Output Distribution: GELU vs ReLU Transformer Block")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"GELU block output -- mean: {out_gelu.mean():.4f}, std: {out_gelu.std():.4f}")
print(f"ReLU block output -- mean: {out_relu.mean():.4f}, std: {out_relu.std():.4f}")

# Examine the intermediate FFN activations (after the activation function)
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

# Get activations inside FFN for GELU block
x_ln2_gelu = block_gelu.ln2(x + block_gelu.attn(block_gelu.ln1(x)))
ffn_hidden_gelu = block_gelu.ffn[0](x_ln2_gelu)  # After first linear
ffn_act_gelu = F.gelu(ffn_hidden_gelu)  # After GELU

# Get activations inside FFN for ReLU block
x_ln2_relu = block_relu.ln2(x + block_relu.attn(block_relu.ln1(x)))
ffn_hidden_relu = block_relu.ffn[0](x_ln2_relu)
ffn_act_relu = F.relu(ffn_hidden_relu)

# Fraction of zeros (dead neurons)
frac_zero_gelu = (ffn_act_gelu.abs() < 1e-6).float().mean().item()
frac_zero_relu = (ffn_act_relu == 0).float().mean().item()

print(f"Fraction of near-zero activations:")
print(f"  GELU: {frac_zero_gelu:.4f} ({frac_zero_gelu * 100:.1f}%)")
print(f"  ReLU: {frac_zero_relu:.4f} ({frac_zero_relu * 100:.1f}%)")
print(f"\nReLU kills {frac_zero_relu * 100:.1f}% of FFN activations vs "
      f"{frac_zero_gelu * 100:.1f}% for GELU.")

ReLU zeroes out a large fraction of the FFN hidden layer, wasting capacity. GELU allows small negative signals to pass through, making better use of the network's parameters.

### 7.4 Ablation Summary: Side-by-Side Comparison

In [ ]:
# Comprehensive ablation: run all variants through N stacked blocks and measure
# gradient norms and output statistics
torch.manual_seed(42)
n_stack = 4

variants = {
    "Full (GPT-2)": [TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(n_stack)],
    "No Residual": [TransformerBlockNoResidual(EMBED_DIM, N_HEADS) for _ in range(n_stack)],
    "No LayerNorm": [TransformerBlockNoNorm(EMBED_DIM, N_HEADS) for _ in range(n_stack)],
    "ReLU (not GELU)": [TransformerBlockReLU(EMBED_DIM, N_HEADS) for _ in range(n_stack)],
}

results = {}
for name, blocks in variants.items():
    model = nn.Sequential(*blocks)
    x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)
    out = model(x)
    out.sum().backward()
    results[name] = {
        "grad_norm": x.grad.norm().item(),
        "out_mean": out.mean().item(),
        "out_std": out.std().item(),
        "out_norm": out.norm().item(),
    }

print(f"{'Variant':<20} {'Grad Norm':>12} {'Out Mean':>12} {'Out Std':>12} {'Out Norm':>12}")
print("-" * 70)
for name, r in results.items():
    print(f"{name:<20} {r['grad_norm']:>12.4f} {r['out_mean']:>12.4f} "
          f"{r['out_std']:>12.4f} {r['out_norm']:>12.4f}")

# Visualize the ablation results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = list(results.keys())
colors = ["#4C72B0", "#C44E52", "#DD8452", "#55A868"]

# Gradient norms
grad_vals = [results[n]["grad_norm"] for n in names]
axes[0].bar(names, grad_vals, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Gradient Norm")
axes[0].set_title("Input Gradient Norms")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(True, alpha=0.3, axis="y")

# Output std
std_vals = [results[n]["out_std"] for n in names]
axes[1].bar(names, std_vals, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Output Std")
axes[1].set_title("Output Standard Deviation")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(True, alpha=0.3, axis="y")

# Output norm
norm_vals = [results[n]["out_norm"] for n in names]
axes[2].bar(names, norm_vals, color=colors, edgecolor="black", linewidth=0.5)
axes[2].set_ylabel("Output L2 Norm")
axes[2].set_title("Output L2 Norm")
axes[2].tick_params(axis="x", rotation=30)
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

The ablation results reveal the contribution of each component:

- **Removing residual connections** typically causes the most dramatic change -- gradients may vanish and output norms can collapse or diverge.
- **Removing LayerNorm** allows activation statistics to drift, leading to larger and more variable output norms.
- **Replacing GELU with ReLU** has a more subtle effect on output statistics, but kills a significant fraction of FFN activations, reducing the network's effective capacity.

---
## Part 8: Stacking Multiple Blocks

GPT-2 Small uses 12 transformer blocks stacked in sequence. Let us simulate this and track how representations evolve.

In [ ]:
# Stack 12 blocks (like GPT-2 Small) and track norms
torch.manual_seed(42)
n_blocks = 12
stack = nn.ModuleList([TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(n_blocks)])

x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
layer_norms_list = [x.norm(dim=-1).mean().item()]
layer_means = [x.mean().item()]
layer_stds = [x.std().item()]

h = x
for i, blk in enumerate(stack):
    h = blk(h)
    layer_norms_list.append(h.norm(dim=-1).mean().item())
    layer_means.append(h.mean().item())
    layer_stds.append(h.std().item())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
layer_idx = list(range(n_blocks + 1))

axes[0].plot(layer_idx, layer_norms_list, "o-", color="#4C72B0", linewidth=2, markersize=5)
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Mean Token Norm")
axes[0].set_title("Token L2 Norm vs Depth")
axes[0].grid(True, alpha=0.3)

axes[1].plot(layer_idx, layer_means, "o-", color="#55A868", linewidth=2, markersize=5)
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Mean Activation")
axes[1].set_title("Activation Mean vs Depth")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].grid(True, alpha=0.3)

axes[2].plot(layer_idx, layer_stds, "o-", color="#C44E52", linewidth=2, markersize=5)
axes[2].set_xlabel("Layer")
axes[2].set_ylabel("Std of Activations")
axes[2].set_title("Activation Std vs Depth")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Input norm: {layer_norms_list[0]:.2f} -> Output norm after 12 blocks: {layer_norms_list[-1]:.2f}")

Even after 12 stacked blocks, the activation norms and statistics remain well-behaved. This is the combined effect of LayerNorm and residual connections -- they make it possible to train networks with dozens or even hundreds of layers.

In [ ]:
# How many parameters would a full GPT-2 Small have?
params_per_block = sum(p.numel() for p in stack[0].parameters())
total_block_params = params_per_block * 12
# GPT-2 Small also has embeddings and a final LN + output head
vocab_size = 50257
max_seq = 1024
embed_params = vocab_size * 768 + max_seq * 768  # token + position embeddings
# In our toy model
print(f"Our toy model:")
print(f"  Params per block: {params_per_block:,}")
print(f"  Total ({n_blocks} blocks): {total_block_params:,}")
print(f"\nGPT-2 Small (for reference):")
print(f"  embed_dim=768, n_heads=12, 12 blocks")
print(f"  ~117M total parameters")

---
## Key Insights

1. **Pre-norm architecture**: GPT-2 applies LayerNorm *before* each sub-layer (attention and FFN), not after. This stabilizes training by ensuring each sub-layer receives normalized inputs, regardless of what upstream layers produce.

2. **LayerNorm**: Normalizes each token's feature vector to zero mean and unit variance, then applies learnable scale and shift. This prevents activation statistics from drifting across layers, which would otherwise make optimization unstable.

3. **GELU activation**: A smooth approximation of ReLU that allows small negative values to pass through. Unlike ReLU, GELU does not kill neurons entirely -- it preserves more information and has smoother gradients, which helps optimization.

4. **Residual connections**: The single most critical component for training deep networks. By adding the input to each sub-layer's output (`x + sublayer(x)`), gradients always have a direct path back through the identity mapping. Without residuals, gradients vanish exponentially with depth.

5. **Feed-forward network**: The 4x expansion (`embed_dim -> 4*embed_dim -> embed_dim`) provides the model with a large hidden space to perform non-linear transformations. The FFN contains roughly two-thirds of each block's parameters.

6. **Component synergy**: Each component plays a distinct role, and removing any one degrades the model:
   - No residuals -> vanishing/exploding gradients
   - No LayerNorm -> activation drift and training instability
   - ReLU instead of GELU -> wasted capacity from dead neurons

7. **Scalability**: The combination of LayerNorm and residual connections allows stacking many blocks (12 in GPT-2 Small, 96 in GPT-3) while maintaining stable activations and healthy gradient flow.